### Attention Mechanism

In [183]:
# Sample mechanism without trainable weights:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [184]:
# use an embedded query token to calculate attention scores:
# Here, we calculate the attention score for the second token 
# with respect to all the other tokens for simplicity
query = inputs[1] # second input                           
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [185]:
# sample
torch.dot(torch.tensor([0.43, 0.15, 0.89]), torch.tensor([0.55, 0.87, 0.66]))

tensor(0.9544)

In [186]:
# Normalize the attention scores,
# these should sum upto 1 (since it is normalized), 
# and this is due to maintain a stability while training LLM's
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [187]:
# Better approach: use softmax (this is a naive version)
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [188]:
# torch's softmax:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [189]:
# calculate the context vector for the 2nd token:
query = inputs[1]        
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


#### Computing Attention weights for all input tokens

In [190]:
# compute the attention scores for all the input tokens
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [191]:
# use built in matrix multiplier:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [192]:
# calculate attention weights:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [193]:
# verify the normalization:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)
print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [194]:
# compute the context vectors:
print(f"{attn_weights},\n{inputs}\n")
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]]),
tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [195]:
# verify the context vector compute vs initial second context vector
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


#### Implementing self-attention with trainable weights

In [196]:
# Computing the attention weights step by step
x_2 = inputs[1]
d_in = inputs.shape[1] # input dimensions = 3
d_out = 2 # output dimensions = 2
# in most GPT-like models, the input and the output dimensions will be same, 
# here we use different for the sake of better computation.

In [197]:
# initialize the query, key and value weight matrices
from torch.nn import Parameter

torch.manual_seed(123)
# requires_grad = True for training, to update the weights,
# here it is set to false to reduce output clutter
# and the weights are frozen when it is set to False.
w_query = Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_key   = Parameter(torch.rand(d_in, d_out), requires_grad=False)
w_value = Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [198]:
# Now, let us compute query, key and value vectors:
query_2 = x_2 @ w_query
key_2   = x_2 @ w_key
value_2 = x_2 @ w_value
print(f"Query for 2nd ip vector: {query_2}\nKey for 2nd ip vector:   {key_2}\nValue for 2nd ip vector: {value_2}")

Query for 2nd ip vector: tensor([0.4306, 1.4551])
Key for 2nd ip vector:   tensor([0.4433, 1.1419])
Value for 2nd ip vector: tensor([0.3951, 1.0037])


In [199]:
# we need to compute the key and value for all the input sequences
keys   = inputs @ w_key
values = inputs @ w_value
print(f"Keys shape: {keys.shape}\nValues shape: {values.shape}") 
# from (6,3) input (input size x embedding dim) projected to (6,2)

Keys shape: torch.Size([6, 2])
Values shape: torch.Size([6, 2])


In [200]:
# now, we should compute the attention score.
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [201]:
# All attention scores for 2
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [202]:
# attention weights now, 
# scaling with the sqrt(d_k), the dimension of keys for a single attention head
# this is done because it helps to control the growth and keeps the spread uniform, 
# the softmax values would be at extremes without this, 
# and if we divide by d_k; almost all the scores comes down to 0, giving a flatline.
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [203]:
# final step to compute context vectors, 
# the weighted sum over the value vectors and normalized attention scores (weights)
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


In [204]:
# Implementing a self-attention class in python:
# nn.Parameter is a learnable tensor.
from torch import nn


class SelfAttentionV1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T 
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
            )
        context_vec = attn_weights @ values
        return context_vec

In [205]:
# Implement this class:
torch.manual_seed(123)
sa_v1 = SelfAttentionV1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [206]:
# Self attention using torch.nn.Linear:
# nn.Linear is a pytorch module that performs linear transformation 
# and has its own parameters (weights and optinal biases).
# nn.Linear has default bias = 1, and scales it by 1/sqrt(d_in)
# stores weights in dimensions of (d_in x d_out)
# qkv_bias 
from torch import nn


class SelfAttentionV2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores/keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [207]:
# implement using SelfAttentionV2.
torch.manual_seed(789)
sa_v2 = SelfAttentionV2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


#### Exercise 3.1

In [208]:
# Exercise 3.1: Comparing weights from SelfAttentionV1 and SelfAttentionV2:
import torch

inputs_exercise = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# input and output dimensions:
e_in, e_out = 3, 2

In [209]:
# implement the SelfAttentionV1 class:
from torch import nn


class SelfAttentionV1Exercise(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = nn.Parameter(torch.rand(d_in,d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(100)
ex_sa_v1 = SelfAttentionV1Exercise(d_in, d_out)

In [210]:
# implement the SelfAttentionV2 class:
from torch import nn


class SelfAttentionV2Exercise(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(100)
ex_sa_v2 = SelfAttentionV2Exercise(d_in, d_out)

In [211]:
ex_sa_v1.W_query = torch.nn.Parameter(ex_sa_v2.W_query.weight.T)
ex_sa_v1.W_key = torch.nn.Parameter(ex_sa_v2.W_key.weight.T)
ex_sa_v1.W_value = torch.nn.Parameter(ex_sa_v2.W_value.weight.T)

In [212]:
ex_sa_v1(inputs_exercise) == ex_sa_v2(inputs_exercise)

tensor([[True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True],
        [True, True]])